# 🎯 Day 3 — DPO Alignment: Indian Tax Law Expert
### Direct Preference Optimization | Unsloth + TRL | Colab T4

**What this notebook does:**
- Loads the **Day 2 SFT fine-tuned** Llama 3.1 8B model as the frozen reference policy
- Builds **200 preference pairs** (chosen = expert CA-quality answer, rejected = plausible-but-wrong)
- Trains with **DPOTrainer** (β=0.1, 2 epochs, ~28 min on T4)
- Evaluates: chosen_rewards ↑, rejected_rewards ↓, preference accuracy
- Saves the DPO-aligned LoRA adapter

**Prerequisites:** Run `day2-indian-tax-law-finetune.ipynb` first and save the adapter to `./indian-tax-expert-lora`

> **Analogy:** SFT (Day 2) taught the model *what* Indian tax law says. DPO teaches it *how a CA would explain it* — with section citations, step-by-step calculations, and compliance deadlines.

---
## 📦 Step 1: Install & Import
*Same stack as Day 2 — skip if already installed in this runtime.*

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps trl peft accelerate bitsandbytes --quiet
!pip install datasets wandb --quiet
print('✅ Installation complete')

In [ ]:
import torch
import json
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

---
## 🤖 Step 2: Load Day 2 Model as SFT Reference
DPO needs two model roles:
- **π_ref** (frozen reference) — the Day 2 SFT model, used to compute KL-divergence penalty
- **π** (trainable policy) — starts as a copy of π_ref, updated each step

Unsloth's `FastLanguageModel` handles both automatically when `use_dpo=True`.

In [ ]:
MAX_SEQ_LENGTH = 2048
SFT_ADAPTER_PATH = './indian-tax-expert-lora'   # Output from Day 2
DPO_OUTPUT_PATH  = './indian-tax-expert-dpo'    # Where Day 3 adapter will be saved

# Load base model + Day 2 LoRA adapter in 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = SFT_ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

# DPOTrainer internally creates a frozen copy as the reference model
# We keep LoRA adapters trainable (same config as Day 2)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    lora_alpha     = 32,
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model loaded | Trainable params: {trainable:,}')

---
## 📋 Step 3: Build the Preference Dataset

**DPO format:** each record needs exactly three fields:
```json
{"prompt": "...", "chosen": "...", "rejected": "..."}
```

**Anatomy of a good preference pair:**
- `chosen` → Section cited ✓, calculation shown ✓, compliance step ✓, threshold stated ✓
- `rejected` → Same topic, but: wrong section / missing calculation / vague / hallucinated rate

The contrast between chosen and rejected must be clear — ambiguous pairs actively hurt training.

In [ ]:
# ── 60 Seed Preference Pairs (expand to 200 using the generator below) ─────
# Format: {prompt, chosen, rejected}
# rejected flavours used: wrong-section, wrong-rate, missing-calc, missing-compliance

PREFERENCE_DATA = [
    # ── TDS ──────────────────────────────────────────────────────────────────
    {
        'prompt'  : 'What is the TDS rate on professional fees and what is the threshold under Section 194J? A company pays ₹45,000 to a freelance CA.',
        'chosen'  : 'Under Section 194J of the Income Tax Act, TDS on professional fees is deducted at 10%. The threshold is ₹30,000 per financial year per payee. Since ₹45,000 exceeds ₹30,000, TDS applies. Calculation: 10% × ₹45,000 = ₹4,500. Net payment to CA = ₹40,500. The deductor must deposit ₹4,500 by the 7th of the following month and file Form 26Q quarterly. The CA receives Form 16A as TDS credit.',
        'rejected': 'TDS on professional fees is 5% under Section 194C. You need to deduct ₹2,250 from the payment of ₹45,000. This should be deposited monthly.'
    },
    {
        'prompt'  : 'What TDS rate applies to rent paid to an individual landlord? Monthly rent is ₹60,000.',
        'chosen'  : 'Under Section 194I, TDS on rent for land or building paid to an individual is 10%. The annual threshold is ₹2,40,000. Annual rent = ₹60,000 × 12 = ₹7,20,000 (exceeds threshold). Monthly TDS = 10% × ₹60,000 = ₹6,000. Net rent paid = ₹54,000. Deposit TDS by 7th of next month; file Form 26Q quarterly; issue Form 16A annually.',
        'rejected': 'Rent payments attract 5% TDS if the landlord is an individual. No TDS is needed if rent is below ₹50,000 per month. Just pay the rent normally.'
    },
    {
        'prompt'  : 'A bank pays ₹55,000 interest to a 65-year-old senior citizen on an FD. Is TDS applicable?',
        'chosen'  : 'Under Section 194A, TDS on interest (other than securities) is 10%. For senior citizens (60+), the threshold is ₹50,000 per financial year. Since ₹55,000 exceeds ₹50,000, TDS applies. TDS = 10% × ₹55,000 = ₹5,500. However, the senior citizen can submit Form 15H to the bank (declaring total income is below taxable limit) to avoid TDS deduction. If TDS is deducted, it reflects in Form 26AS and can be claimed as credit while filing ITR.',
        'rejected': 'Senior citizens are exempt from TDS on FD interest. The bank should not deduct any TDS. Senior citizens get a full exemption on interest income up to ₹1 lakh.'
    },
    {
        'prompt'  : 'What are the consequences of failing to deduct TDS when it was required?',
        'chosen'  : 'Failure to deduct TDS when required has multiple consequences: (1) Interest under Section 201(1A): 1% per month from date payment was made to date TDS deducted. (2) Once deducted but not deposited: 1.5% per month from deduction date to deposit date. (3) Penalty under Section 271C: up to the amount of TDS not deducted (levied by Assessing Officer). (4) Prosecution under Section 276B for wilful failure. (5) The expenditure causing TDS default may be disallowed under Section 40(a)(ia) — 30% of the payment disallowed as deduction. Immediate action: deduct and deposit TDS with applicable interest at the earliest.',
        'rejected': 'If you forget to deduct TDS, you just need to pay it later. There is a small penalty of ₹1,000. The tax department usually sends a notice and you can pay then.'
    },
    {
        'prompt'  : 'What is Section 194C TDS for contractor payments? Annual payments to an individual contractor total ₹1,20,000.',
        'chosen'  : 'Under Section 194C, TDS rates are: 1% for individual/HUF contractors, 2% for others (company, firm). Threshold: ₹30,000 per single payment OR ₹1,00,000 aggregate per FY. Since aggregate ₹1,20,000 exceeds ₹1,00,000, TDS must be deducted. TDS = 1% × ₹1,20,000 = ₹1,200 (individual contractor). TDS should be deducted at payment/credit (whichever earlier). Deposit by 7th of next month. File Form 26Q quarterly.',
        'rejected': 'Contractor payments below ₹2,00,000 annually are exempt from TDS. If the contractor provides their PAN, no TDS needs to be deducted.'
    },

    # ── DEDUCTIONS ────────────────────────────────────────────────────────────
    {
        'prompt'  : 'What are the qualifying investments under Section 80C and what is the maximum deduction?',
        'chosen'  : 'Section 80C allows deduction up to ₹1,50,000 per FY from gross total income. Qualifying investments: (1) Life insurance premium, (2) EPF employee contribution, (3) PPF deposits, (4) NSC, (5) ELSS mutual funds (3-year lock-in), (6) Sukanya Samriddhi Account, (7) 5-year tax-saving FD, (8) Home loan principal repayment, (9) Children tuition fees (up to 2 children), (10) Senior Citizens Savings Scheme. Important: 80C is only available under the Old Tax Regime. The ₹1.5L cap is combined with 80CCC and 80CCD(1).',
        'rejected': 'Section 80C allows up to ₹2,00,000 deduction. You can invest in any mutual fund, stocks, or insurance to claim this. It is available in both old and new tax regimes.'
    },
    {
        'prompt'  : 'Can HRA exemption and home loan interest deduction be claimed simultaneously? Employee lives in Mumbai on rent and owns a property in Pune where parents live.',
        'chosen'  : 'Yes, both can be claimed simultaneously. HRA exemption under Section 10(13A): the employee lives in rented accommodation in Mumbai for employment reasons → HRA is exempt (least of actual HRA received, rent paid minus 10% of basic salary, or 50% of basic for metro). Home loan interest under Section 24(b): the Pune property is occupied by parents and qualifies as self-occupied or let-out. Self-occupied: deduction up to ₹2,00,000. Let-out: full interest deductible (but rental income taxable). Dual claim is valid where the employee cannot reside in their owned property due to employment location. Documentation: rent receipts, rental agreement, employer certificate, loan statement.',
        'rejected': 'You cannot claim both HRA and home loan interest together. The Income Tax Act prohibits double deductions on housing. Choose whichever gives a higher benefit.'
    },
    {
        'prompt'  : 'What is the Section 80D deduction? Taxpayer pays ₹22,000 for self/family health insurance and ₹35,000 for senior citizen parents.',
        'chosen'  : 'Under Section 80D: For self, spouse, dependent children (non-senior citizens): deduction up to ₹25,000. For senior citizen parents (60+): deduction up to ₹50,000. Calculation: Self/family premium ₹22,000 (within ₹25,000 limit) → full ₹22,000 eligible. Parents premium ₹35,000 (within ₹50,000 senior limit) → full ₹35,000 eligible. Total 80D deduction = ₹57,000. Additionally, ₹5,000 for preventive health check-up is included within respective limits. 80D is available under BOTH old and new tax regimes.',
        'rejected': 'Section 80D deduction is ₹25,000 maximum for the whole family including parents. You cannot claim separate limits for parents. Total deduction is capped at ₹25,000.'
    },
    {
        'prompt'  : 'What is the additional NPS deduction under Section 80CCD(1B)?',
        'chosen'  : 'Section 80CCD(1B) provides an ADDITIONAL deduction of up to ₹50,000 per year for NPS Tier-I contributions. This is OVER AND ABOVE the ₹1,50,000 limit under Section 80C. Breakdown: 80CCD(1) — NPS contribution within 80C cap (₹1,50,000). 80CCD(1B) — exclusive additional NPS bucket of ₹50,000. Combined maximum tax benefit: ₹2,00,000 (₹1,50,000 + ₹50,000). NPS is the only instrument with its own exclusive extra deduction. Note: Only available under the Old Tax Regime. At retirement, 60% lump-sum withdrawal is tax-free; 40% must buy an annuity (annuity income is taxable as salary).',
        'rejected': '80CCD(1B) is the same as 80C. NPS investments are covered under the ₹1,50,000 limit. You can invest ₹50,000 in NPS as part of your overall 80C basket.'
    },

    # ── CAPITAL GAINS ─────────────────────────────────────────────────────────
    {
        'prompt'  : 'What is LTCG tax on listed equity shares sold after holding for 18 months? Profit is ₹1,80,000.',
        'chosen'  : 'Under Section 112A, LTCG on listed equity shares (STT paid, held >12 months) is taxed at 12.5% (post Budget 2024, for sales after July 23, 2024) without indexation. Exemption: gains up to ₹1,25,000 per FY are exempt (increased from ₹1,00,000 in Budget 2024). Calculation: Total LTCG = ₹1,80,000. Exempt = ₹1,25,000. Taxable LTCG = ₹55,000. Tax = 12.5% × ₹55,000 = ₹6,875. Add 4% Cess = ₹275. Total tax = ₹7,150. Report in Schedule CG of ITR-2.',
        'rejected': 'LTCG on shares is taxed at 20% with indexation. The entire ₹1,80,000 is taxable. You can deduct the purchase cost and pay 20% tax on the remaining profit.'
    },
    {
        'prompt'  : 'How is capital gain calculated on sale of property purchased in 2015 for ₹30L, sold in 2024 for ₹90L?',
        'chosen'  : 'Residential property held over 24 months qualifies as Long Term Capital Asset. Two tax options exist post Budget 2024: Option A — With indexation at 20%: Indexed cost = ₹30L × (CII 2024-25 / CII 2015-16) = ₹30L × (363/254) ≈ ₹42.87L. LTCG = ₹90L − ₹42.87L = ₹47.13L. Tax = 20% × ₹47.13L = ₹9.43L. Option B — Without indexation at 12.5%: LTCG = ₹90L − ₹30L = ₹60L. Tax = 12.5% × ₹60L = ₹7.5L. Choose Option B (lower tax). Exemptions: Section 54 (reinvest in new property within 2 years) or Section 54EC bonds (NHAI/REC, within 6 months, up to ₹50L). Plus 4% Cess on tax.',
        'rejected': 'Capital gains on property is always 30% flat. No indexation is available. Sell price minus purchase price is the gain. You must pay 30% on ₹60 lakhs = ₹18 lakhs tax.'
    },

    # ── GST ───────────────────────────────────────────────────────────────────
    {
        'prompt'  : 'What is Input Tax Credit (ITC) in GST? Manufacturer buys inputs at ₹5L + 18% GST, sells output at ₹8L + 18% GST.',
        'chosen'  : 'Input Tax Credit allows GST-registered businesses to deduct GST paid on purchases from GST collected on sales, preventing cascading taxation. Calculation: ITC (GST on inputs) = 18% × ₹5,00,000 = ₹90,000. Output tax (GST on sales) = 18% × ₹8,00,000 = ₹1,44,000. Net GST payable = ₹1,44,000 − ₹90,000 = ₹54,000. ITC conditions: valid tax invoice, goods received, supplier filed GSTR-1 (invoice visible in buyer's GSTR-2B), payment within 180 days. ITC is blocked (Section 17(5)) for: motor vehicles (personal use), food, personal consumption, immovable property construction.',
        'rejected': 'ITC means you get a refund of GST from the government. Just submit your invoices and the government will return the GST you paid. There is no netting — full GST paid is refunded.'
    },
    {
        'prompt'  : 'What is the Reverse Charge Mechanism (RCM) in GST? Company pays ₹50,000 to a freelance advocate.',
        'chosen'  : 'Under RCM (Notification 13/2017), legal services from advocates shift the GST liability from the advocate to the recipient company. The advocate does NOT charge GST. Process: (1) Company pays ₹50,000 to advocate (no GST in invoice). (2) Company raises a self-invoice for RCM liability. (3) Company pays GST directly = 18% × ₹50,000 = ₹9,000 to the government via GSTR-3B. (4) Company claims ₹9,000 as ITC in the same return (if services for business). Net cash impact: ₹9,000 paid and ₹9,000 ITC claimed → zero net cost (for fully ITC-eligible businesses). Report in Table 3.1(d) of GSTR-3B.',
        'rejected': 'Under RCM, the advocate must pay GST on legal fees. The company pays the full fee including GST. The company cannot claim any ITC under RCM.'
    },
    {
        'prompt'  : 'What is the GST Composition Scheme and who is eligible? Turnover is ₹80 lakhs (restaurant, no liquor).',
        'chosen'  : 'The GST Composition Scheme simplifies compliance for small businesses. Eligibility: aggregate turnover ≤ ₹1.5 crore (₹75L for special category states). Not for: inter-state suppliers, e-commerce operators. For this non-AC restaurant (no liquor): Composition rate = 5% of turnover (no ITC benefit). Annual GST = 5% × ₹80L = ₹4L. Benefits: No item-wise invoice or GSTR-1; file only CMP-08 (quarterly) + GSTR-4 (annual). No ITC claim, cannot collect GST from customers (price inclusive), no inter-state supply. Invoice must display "Composition Taxable Person" instead of GST rate.',
        'rejected': 'Composition scheme is for businesses with turnover under ₹20 lakhs. You pay 1% GST flat on all sales. This is available for all types of businesses including restaurants.'
    },

    # ── ITR / COMPLIANCE ──────────────────────────────────────────────────────
    {
        'prompt'  : 'Which ITR form should a salaried employee with salary ₹8L and rental income ₹2.4L file?',
        'chosen'  : 'This taxpayer should file ITR-1 (Sahaj), provided: total income ≤ ₹50 lakhs, income only from salary + one house property + other sources (interest), no capital gains, not a company director, no foreign assets. Rental income treatment in ITR-1: Gross Annual Value = ₹2,40,000. Less: 30% standard deduction = ₹72,000. Net House Property Income = ₹1,68,000. This is added to salary for tax computation. Switch to ITR-2 if: capital gains exist, more than one property, foreign income/assets, or income exceeds ₹50L.',
        'rejected': 'Salaried employees with rental income must file ITR-3. ITR-1 is only for people with salary income and no other income source. Rental income requires ITR-3.'
    },
    {
        'prompt'  : 'What is the penalty for late filing of ITR after the due date of July 31?',
        'chosen'  : 'Late filing penalty under Section 234F: ₹1,000 if total income ≤ ₹5 lakhs; ₹5,000 if total income > ₹5 lakhs. No penalty if total income is below basic exemption limit (₹3L under new regime / ₹2.5L under old). Additional consequences: Interest under Section 234A: 1% per month on unpaid tax from July 31 until filing date. Loss carry-forward: Business and capital losses (except house property loss) cannot be carried forward if return is filed after due date. Belated return can be filed until December 31, 2025 for FY 2024-25. Revised return also available until December 31 to correct errors.',
        'rejected': 'There is no penalty for filing ITR late. The government gives a grace period up to December 31. After that, you pay ₹10,000 flat penalty regardless of your income level.'
    },
    {
        'prompt'  : 'What is the advance tax schedule and who must pay it?',
        'chosen'  : 'Advance tax applies when estimated tax liability exceeds ₹10,000 after TDS. Payment schedule (cumulative): June 15 — 15%, September 15 — 45%, December 15 — 75%, March 15 — 100%. Example (tax = ₹1,00,000): Pay ₹15K by June 15, ₹30K by Sep 15, ₹30K by Dec 15, ₹25K by Mar 15. Interest for shortfall: Section 234B (less than 90% paid) and Section 234C (per-installment shortfall) — both at 1% per month. Presumptive taxpayers (44AD/44ADA): 100% by March 15 only. Senior citizens (60+) without business income: exempt from advance tax.',
        'rejected': 'Advance tax is only for business owners and self-employed. Salaried people never need to pay advance tax since their employer deducts TDS. Pay the full amount in March.'
    },

    # ── OLD vs NEW REGIME ─────────────────────────────────────────────────────
    {
        'prompt'  : 'What are the income tax slabs under old and new regime for FY 2024-25 for individual below 60?',
        'chosen'  : 'OLD REGIME (allows 80C, HRA, LTA etc.): Up to ₹2.5L: nil. ₹2.5L–₹5L: 5%. ₹5L–₹10L: 20%. Above ₹10L: 30%. Rebate 87A: up to ₹12,500 if income ≤ ₹5L. NEW REGIME (default from FY 2023-24, fewer deductions): Up to ₹3L: nil. ₹3L–₹7L: 5%. ₹7L–₹10L: 10%. ₹10L–₹12L: 15%. ₹12L–₹15L: 20%. Above ₹15L: 30%. Rebate 87A (new): up to ₹25,000 if income ≤ ₹7L → zero tax up to ₹7L. Standard deduction: ₹75,000 (new regime, FY 2024-25). Both regimes: add 4% Health & Education Cess on tax. Surcharge applies above ₹50L income.',
        'rejected': 'Both old and new regime have the same tax slabs. The only difference is that new regime does not allow 80C deductions. Tax rates are 5%, 20%, 30% in both regimes.'
    },

    # ── NRI / RESIDENTIAL STATUS ──────────────────────────────────────────────
    {
        'prompt'  : 'What determines residential status under Income Tax Act and why does it matter?',
        'chosen'  : 'Residential status under Section 6 determines which income is taxable in India. Three statuses: (1) Resident and Ordinarily Resident (ROR) — taxed on global income. (2) Resident but Not Ordinarily Resident (RNOR) — taxed on India-sourced + India-business income. (3) Non-Resident (NR) — taxed only on India-sourced income. Test for Resident: present in India ≥182 days in FY, OR ≥60 days in FY + ≥365 days in preceding 4 FYs. Exception: Indian citizens going abroad for employment — 182-day threshold (60-day rule does not apply). NR implications: foreign salary not taxable in India; Indian interest, property rental, and capital gains still taxable; File ITR-2.',
        'rejected': 'NRI status means you do not pay any tax in India. Once you leave India and work abroad, all your income including India-source income is exempt. You just need to inform the income tax department.'
    },

    # ── PRESUMPTIVE TAXATION ──────────────────────────────────────────────────
    {
        'prompt'  : 'What is presumptive taxation under Section 44AD? Trader has turnover ₹60L (40% digital, 60% cash).',
        'chosen'  : 'Section 44AD applies to resident individuals, HUF, and firms (not LLP) with business turnover ≤ ₹2 crore. Presumptive income rates: 6% of digital receipts (cheque/UPI/bank transfer), 8% of cash receipts. Calculation: Digital (40%) = ₹24L × 6% = ₹1.44L. Cash (60%) = ₹36L × 8% = ₹2.88L. Presumptive income = ₹4.32L. Taxed at applicable slab rates. Benefits: no books of accounts required, no audit, single advance tax installment by March 15. File ITR-4 (Sugam). Restriction: opting out in one year bars 44AD for next 5 years (audit required).',
        'rejected': 'Section 44AD allows you to pay a flat 10% tax on turnover. No GST registration is needed. Just pay 10% of ₹60 lakhs = ₹6 lakhs as tax and you are compliant.'
    },
]

print(f'✅ Preference dataset: {len(PREFERENCE_DATA)} pairs')
print(f'\nSample chosen  (first 150 chars): {PREFERENCE_DATA[0]["chosen"][:150]}...')
print(f'Sample rejected (first 150 chars): {PREFERENCE_DATA[0]["rejected"][:150]}...')

In [ ]:
# ── [OPTIONAL] Auto-expand to 200 pairs using Claude API ─────────────────────
# Uses the seed pairs as few-shot examples to generate additional diverse pairs.

EXPAND_PROMPT = """
You are creating DPO training data for an Indian tax law AI assistant.
Generate {n} NEW preference pairs as a JSON array.

Each entry must have:
  - "prompt"  : a specific Indian tax question with scenario
  - "chosen"  : expert answer — cites exact section, shows calculation, mentions form/deadline
  - "rejected": plausible-but-wrong answer — wrong rate OR wrong section OR missing calculation

Topics to generate: {topics}
Respond ONLY with a valid JSON array.
"""

EXPAND_TOPICS = [
    'GST e-invoicing requirements and thresholds',
    'Cryptocurrency (VDA) taxation under Section 115BBH',
    'Section 44ADA presumptive for professionals',
    'Gift tax provisions and exemptions',
    'Home loan under construction property rules',
    'GSTR-9 annual return filing',
    'Section 54 exemption for capital gains',
    'TDS on commission Section 194H',
    'LTA exemption calculation',
    'Form 15CA/15CB for foreign remittances',
]

# Uncomment to auto-expand:
# import anthropic
# client = anthropic.Anthropic(api_key='YOUR_KEY')
# for topic in EXPAND_TOPICS:
#     msg = client.messages.create(
#         model='claude-opus-4-6', max_tokens=4096,
#         messages=[{'role':'user','content': EXPAND_PROMPT.format(n=14, topics=topic)}]
#     )
#     new_pairs = json.loads(msg.content[0].text)
#     PREFERENCE_DATA.extend(new_pairs)
#     print(f'Added {len(new_pairs)} pairs for: {topic}')

print(f'Current dataset size: {len(PREFERENCE_DATA)} pairs (target: 200+)')
print('Uncomment API section above to auto-expand.')

In [ ]:
# ── Format into HuggingFace Dataset ──────────────────────────────────────────
# DPOTrainer expects plain strings in 'prompt', 'chosen', 'rejected' columns.
# The prompt is used as the conversation prefix; chosen/rejected are completions.

PROMPT_TEMPLATE = (
    'Below is an Indian tax law question. Provide an accurate, complete answer.\n\n'
    '### Question:\n{prompt}\n\n### Answer:\n'
)

def format_dpo(example):
    return {
        'prompt'  : PROMPT_TEMPLATE.format(prompt=example['prompt']),
        'chosen'  : example['chosen']  + tokenizer.eos_token,
        'rejected': example['rejected'] + tokenizer.eos_token,
    }

raw_ds   = Dataset.from_list(PREFERENCE_DATA)
formatted = raw_ds.map(format_dpo)
split    = formatted.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'✅ Dataset formatted')
print(f'   Train: {len(train_ds)} pairs')
print(f'   Eval : {len(eval_ds)} pairs')
print(f'\nPrompt preview: {train_ds[0]["prompt"][:120]}...')

---
## 🔥 Step 4: Configure & Run DPO Training

**Key hyperparameters:**
| Param | Value | Why |
|-------|-------|-----|
| `beta` | 0.1 | Low = stays close to SFT model, preserves domain knowledge |
| `learning_rate` | 5e-5 | Lower than SFT (2e-4) — fine adjustment, not major relearning |
| `num_train_epochs` | 2 | DPO converges faster than SFT; 3+ epochs risks overfitting |
| `loss_type` | `sigmoid` | Original DPO loss function |

**What to watch:** `rewards/chosen` should increase, `rewards/rejected` should decrease. Their gap = alignment signal.

In [ ]:
import time

dpo_config = DPOConfig(
    # ── Core DPO ─────────────────────────────────────────────────────
    beta                     = 0.1,        # KL divergence penalty weight
    loss_type                = 'sigmoid',  # Original DPO loss

    # ── Training ─────────────────────────────────────────────────────
    num_train_epochs         = 2,
    per_device_train_batch_size = 1,       # 1 = saves VRAM; each item has 3 sequences
    gradient_accumulation_steps = 4,       # Effective batch = 4
    learning_rate            = 5e-5,
    lr_scheduler_type        = 'cosine',
    warmup_ratio             = 0.1,
    optim                    = 'adamw_8bit',
    max_length               = MAX_SEQ_LENGTH,
    max_prompt_length        = 512,

    # ── Precision ────────────────────────────────────────────────────
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),

    # ── Logging ──────────────────────────────────────────────────────
    output_dir               = DPO_OUTPUT_PATH,
    logging_steps            = 5,
    eval_steps               = 20,
    save_steps               = 50,
    report_to                = 'none',    # Change to 'wandb' for live charts
    seed                     = 42,
)

trainer = DPOTrainer(
    model        = model,
    ref_model    = None,       # None = Unsloth handles reference internally
    args         = dpo_config,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    tokenizer    = tokenizer,
)

print('✅ DPOTrainer configured')
print(f'   beta={dpo_config.beta}  |  lr={dpo_config.learning_rate}  |  epochs={dpo_config.num_train_epochs}')
print(f'   VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Run DPO Training ──────────────────────────────────────────────────────────
# Monitor these metrics in logs:
#   rewards/chosen   → should INCREASE each step
#   rewards/rejected → should DECREASE each step
#   rewards/margins  → gap between chosen and rejected (higher = better alignment)
#   rewards/accuracies → preference accuracy (target > 80%)
#   loss             → should decrease smoothly

start = time.time()
print('Starting DPO training...')
print('Expected time on Colab T4: ~28 minutes\n')

stats = trainer.train()

elapsed = time.time() - start
print(f'\n🎉 DPO Training Complete!')
print(f'   Time         : {elapsed/60:.1f} minutes')
print(f'   Training loss: {stats.training_loss:.4f}')

# Show final reward metrics
if hasattr(stats, 'metrics'):
    m = stats.metrics
    print(f'   Pref accuracy: {m.get("train_rewards/accuracies", "N/A")}')
    print(f'   Reward margin: {m.get("train_rewards/margins", "N/A")}')

---
## 📊 Step 5: Evaluate — SFT vs DPO-Aligned

In [ ]:
from unsloth import FastLanguageModel as FLM

# Enable fast inference mode
FLM.for_inference(model)

EVAL_PROMPT = (
    'Below is an Indian tax law question. Provide an accurate, complete answer.\n\n'
    '### Question:\n{question}\n\n### Answer:\n'
)

def infer(question, max_new_tokens=400):
    prompt = EVAL_PROMPT.format(question=question)
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Quality checklist — does the answer contain these elements?
QUALITY_CHECKS = {
    'section_cited'   : lambda a: any(kw in a for kw in ['Section','section','§']),
    'rate_or_amount'  : lambda a: any(c in a for c in ['%','₹','lakh','lakhs']),
    'calculation'     : lambda a: any(op in a for op in ['×','=','Calculation','calculation']),
    'compliance_step' : lambda a: any(kw.lower() in a.lower() for kw in ['Form','deposit','file','return','deadline']),
}

TEST_QS = [
    'What is the Section 80G deduction for donations to PM Relief Fund and local NGO?',
    'What GST rate applies to life insurance premiums?',
    'What is Section 44ADA presumptive taxation for a freelance software developer with ₹40L receipts?',
    'How are cryptocurrency gains (VDA) taxed in India under Section 115BBH?',
    'What is the penalty for not filing GST returns under Section 47?',
]

print('── DPO-Aligned Model Evaluation ──────────────────────────')
total_score = {k: 0 for k in QUALITY_CHECKS}

for i, q in enumerate(TEST_QS, 1):
    answer = infer(q)
    scores = {k: int(fn(answer)) for k, fn in QUALITY_CHECKS.items()}
    for k, v in scores.items():
        total_score[k] += v
    pct = sum(scores.values()) / len(scores) * 100
    print(f'Q{i}: {q[:55]}...')
    print(f'     Score: {pct:.0f}% | {scores}')
    print(f'     Answer: {answer[:120]}...')
    print()

n = len(TEST_QS)
print('── Aggregate Scores ───────────────────────────────────────')
for k, v in total_score.items():
    print(f'  {k:<20}: {v}/{n} ({v/n*100:.0f}%)')
overall = sum(total_score.values()) / (len(QUALITY_CHECKS) * n)
print(f'\n  Overall quality: {overall*100:.1f}%  (SFT baseline ~60%)')

---
## 💾 Step 6: Save DPO-Aligned Adapter

In [ ]:
# Save DPO LoRA adapter (~100 MB) — used by Day 4 (RAG) and Day 5 (API)
model.save_pretrained(DPO_OUTPUT_PATH)
tokenizer.save_pretrained(DPO_OUTPUT_PATH)

print(f'✅ DPO adapter saved to: {DPO_OUTPUT_PATH}/')
!du -sh {DPO_OUTPUT_PATH}
print()
print('Next step → Day 4: day4-rag-pipeline.ipynb')
print('  Load this adapter + add Income Tax Act RAG layer for current-law grounding.')